# 01. RAG 에이전트 — 벡터 검색 기반 질의응답

## 학습 목표

- InMemoryVectorStore로 벡터 검색 파이프라인을 구축한다
- `content_and_artifact` 반환 형식으로 검색 도구를 정의한다
- `create_deep_agent`로 RAG 에이전트를 생성하고 질의한다
- v1 미들웨어(ModelCallLimitMiddleware, ToolRetryMiddleware)를 적용한다
- **Skills 시스템**으로 RAG 도메인 지식을 점진적 공개(Progressive Disclosure)한다

## 개요

| 항목 | 내용 |
|------|------|
| **프레임워크** | LangChain + Deep Agents |
| **핵심 컴포넌트** | InMemoryVectorStore, OpenAIEmbeddings, RecursiveCharacterTextSplitter |
| **에이전트 패턴** | `content_and_artifact` 도구 → `create_deep_agent` |
| **백엔드** | `FilesystemBackend(root_dir=".", virtual_mode=True)` |
| **스킬** | `skills/rag-agent/SKILL.md` — RAG 도메인 지식 점진적 공개 |

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요"


In [2]:
# Observability 설정 (선택)
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## 1단계: 샘플 문서 생성

RAG 파이프라인의 첫 단계는 검색 대상 문서를 준비하는 것입니다. 실제 환경에서는 PDF, 웹 페이지, 데이터베이스 등에서 문서를 로드하지만, 여기서는 학습 목적으로 직접 `Document` 객체를 생성합니다.


In [4]:
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain은 LLM 애플리케이션 개발 프레임워크입니다. 도구, 체인, 에이전트를 지원합니다.", metadata={"source": "langchain"}),
    Document(page_content="LangGraph는 상태 기반 워크플로를 구축하는 프레임워크입니다. 그래프 API와 Functional API를 제공합니다.", metadata={"source": "langgraph"}),
    Document(page_content="Deep Agents는 올인원 에이전트 SDK입니다. create_deep_agent로 에이전트를 생성하고, 백엔드와 서브에이전트를 지원합니다.", metadata={"source": "deepagents"}),
    Document(page_content="RAG는 검색 증강 생성의 약자로, 외부 지식을 LLM에 주입하여 정확한 응답을 생성합니다.", metadata={"source": "rag"}),
    Document(page_content="벡터 스토어는 임베딩을 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma 등이 있습니다.", metadata={"source": "vectorstore"}),
    Document(page_content="에이전트는 LLM이 도구를 사용하여 자율적으로 작업을 수행하는 시스템입니다. ReAct 패턴이 대표적입니다.", metadata={"source": "agent"}),
]
print(f"문서 {len(docs)}개 생성 완료")


문서 6개 생성 완료


## 2단계: 텍스트 분할

큰 문서를 검색에 적합한 크기의 청크로 분할합니다. `RecursiveCharacterTextSplitter`는 단락 → 문장 → 단어 순으로 자연스러운 경계에서 분할을 시도합니다.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=50
)
splits = splitter.split_documents(docs)
print(f"분할 결과: {len(splits)}개 청크")


분할 결과: 6개 청크

## 3단계: 벡터 스토어 구축

OpenAI 임베딩 모델로 텍스트를 벡터로 변환하고, `InMemoryVectorStore`에 저장합니다. 프로덕션에서는 FAISS나 Chroma 같은 영구 저장소를 사용합니다.


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)
print(f"벡터 스토어 구축 완료 — {len(splits)}개 문서 임베딩됨")


벡터 스토어 구축 완료 — 6개 문서 임베딩됨


## 4단계: 검색 도구 정의 (content_and_artifact)

`response_format="content_and_artifact"` 패턴은 도구가 두 가지를 반환하게 합니다:
- **content**: 에이전트에게 보여줄 텍스트 요약
- **artifact**: 전체 Document 객체 (후속 처리용)

이 패턴은 에이전트의 컨텍스트를 절약하면서도 원본 데이터에 접근할 수 있게 합니다.


In [7]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """벡터 스토어에서 관련 문서를 검색합니다."""
    results = vectorstore.similarity_search(query, k=3)
    content = "\n\n".join(d.page_content for d in results)
    return content, results


## 5단계: 검색 도구 단독 테스트

에이전트에 연결하기 전에 도구가 올바르게 동작하는지 확인합니다.


In [8]:
result = retrieve.invoke({"query": "에이전트란 무엇인가?"})
print(result)


에이전트는 LLM이 도구를 사용하여 자율적으로 작업을 수행하는 시스템입니다. ReAct 패턴이 대표적입니다.

Deep Agents는 올인원 에이전트 SDK입니다. create_deep_agent로 에이전트를 생성하고, 백엔드와 서브에이전트를 지원합니다.

벡터 스토어는 임베딩을 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma 등이 있습니다.


## 6단계: RAG 에이전트 생성 (v1 미들웨어 적용)

에서 프롬프트를 로드합니다. LangSmith Hub → Langfuse → 기본값 순으로 시도합니다.

| 미들웨어 | 역할 |
|---------|------|
| \ | 무한 루프 방지 — 최대 모델 호출 횟수 제한 |
| \ | 검색 도구 실패 시 자동 재시도 |

In [9]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolRetryMiddleware,
)
from prompts import RAG_AGENT_PROMPT

agent = create_deep_agent(
    model=model,
    tools=[retrieve],
    system_prompt=RAG_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),
        ToolRetryMiddleware(max_retries=2),
    ],
)

Prompt 'rag-agent-label:production' not found during refresh, evicting from cache.


Prompt 'sql-agent-label:production' not found during refresh, evicting from cache.


Prompt 'data-analysis-agent-label:production' not found during refresh, evicting from cache.


Prompt 'ml-agent-label:production' not found during refresh, evicting from cache.


Prompt 'deep-research-agent-label:production' not found during refresh, evicting from cache.


## 7단계: 단순 질의 및 비교 질의

단순 질의(하나의 검색)와 비교 질의(다중 검색)를 통해 에이전트의 RAG 동작을 확인합니다.


In [10]:
# 단순 질의
response = agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph가 뭔가요?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)


LangGraph는 상태 기반 워크플로를 구축하기 위한 프레임워크입니다. 그래프 API와 Functional API를 통해 복잡한 LLM(대형 언어 모델) 기반 애플리케이션의 흐름을 정의하고 관리할 수 있습니다.

출처: 검색 결과(내부 문서 검색)


## 요약

| 항목 | 핵심 |
|------|------|
| **벡터 스토어** | `InMemoryVectorStore.from_documents()` — 임베딩 기반 유사도 검색 |
| **검색 도구** | `@tool(response_format="content_and_artifact")` — 요약 + 원본 분리 |
| **에이전트** | `create_deep_agent(model, tools=[retrieve], backend=..., skills=["/skills/"])` |
| **스킬** | `skills/rag-agent/SKILL.md` — Progressive Disclosure로 토큰 절약 |

---

**참고 문서:**
- `docs/langchain/24-retrieval.md`
- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- `docs/deepagents/10-skills.md`

**다음 단계:** → [02_sql_agent.ipynb](./02_sql_agent.ipynb): SQL 에이전트를 구축합니다.